In [1]:
from nltkParser import words_to_pos_vector
from mongo_utils import get_mongo_client
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from machine_learning import perform_kmeans, perform_hierarchical, run_DBSCAN, Gaussian_Mixture_Models, Isolation_Forest

Get Data from MongoDB

In [2]:
import os
os.environ["MONGODB_ATLAS_PASSWORD"] = "TQrNN8PulZs6y5oh"

In [3]:
# Initialize the MongoDB Atlas Connection
username = 'Eric'
cluster_url = 'cluster0.figrf53.mongodb.net'
db_name = 'newsArticle'

mongo_client = get_mongo_client(username, cluster_url, db_name)
db = mongo_client[db_name]
topicsForAI = db['topicsForAI']
chatGPT_articles = db['ChatGPT-Articles']
pro_quest_articles = db['ProQuest-Articles']
nyTimes_articles = db['NYTimeArticle']

In [4]:
# print the length of the collections
print(f"ChatGPT-Articles: {chatGPT_articles.count_documents({})}")
print(f"ProQuest-Articles: {pro_quest_articles.count_documents({})}")
print(f"NYTimeArticle: {nyTimes_articles.count_documents({})}")

ChatGPT-Articles: 398
ProQuest-Articles: 1900
NYTimeArticle: 191


In [5]:
def fetch_embedded_vector(collection):
    # Fetch only the necessary fields
    documents = collection.find({}, {'_id': 1, 'Model Used': 1, 'text-embedding-3-small': 1})
    results = {}
        
    for doc in documents:

        if collection.name == 'NYTimeArticle':
            type_id = ('NYT', str(doc['_id']))
        else:
            # Using 'Model Used' or 'human' if it doesn't exist
            type_id = (doc.get('Model Used', 'human'), str(doc['_id']))
        # Directly assign the array of floats to the tuple key
        results[type_id] = doc.get('text-embedding-3-small', [])
    return results

def fetch_pos_vector(collection):
    # Fetch only the necessary fields
    documents = collection.find({}, {'_id': 1, 'Model Used': 1, 'wordFamilyContent': 1})
    results = {}
    
    for doc in documents:
        if collection.name == 'NYTimeArticle':
            type_id = ('NYT', str(doc['_id']))
        else:
            # Using 'Model Used' or 'human' if it doesn't exist
            type_id = (doc.get('Model Used', 'human'), str(doc['_id']))
        # Directly assign the array of floats to the tuple key
        results[type_id] = words_to_pos_vector(doc.get('wordFamilyContent', []))
    return results

def fetch_title_and_content(collection):
    # Fetch only the necessary fields
    documents = collection.find({}, {'_id': 1, 'Model Used': 1, 'Title': 1, 'Content':1})
    results = {}
    for doc in documents:
        if collection.name == 'NYTimeArticle':
            type_id = ('NYT', str(doc['_id']))
        else:
            # Using 'Model Used' or 'human' if it doesn't exist
            type_id = (doc.get('Model Used', 'human'), str(doc['_id']))
        # Directly assign the content to the tuple key
        content = doc.get('Title') + '\n' + doc.get('Content')
        results[type_id] = content
    return results
    

In [6]:
from sklearn.decomposition import PCA


def plot_clusters(labeled_data, graph_title='Cluster Visualization'):
    # Prepare the data for plotting
    types = [str(key[0]) for key in labeled_data.keys()]
    labels = [data['cluster'] for data in labeled_data.values()]
    vectors = [data['vector'] for data in labeled_data.values()]

    # Converting list of vectors into a 2D array for plotting
    vectors = np.array(vectors)

    # PCA Reduction if necessary
    if vectors.shape[1] > 2:
        pca = PCA(n_components=2)
        vectors = pca.fit_transform(vectors)
        print("Data has been reduced to 2 dimensions using PCA for visualization.")

    # Create unique combinations of type and cluster for markers and colors
    unique_types = sorted(set(types))
    unique_clusters = sorted(set(labels))
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_clusters)))  # Color map for types
    markers = ['o', 's', '^', 'D']  # Extend this list if you have more clusters

    cluster_to_color = {c: col for c, col in zip(unique_clusters, colors)}
    type_to_marker = {t: m for t, m in zip(unique_types, markers)}

    # Create the scatter plot
    plt.figure(figsize=(24, 12))
    for type_val, label, vector in zip(types, labels, vectors):
        color = cluster_to_color[label]
        marker = type_to_marker[type_val]
        plt.scatter(vector[0], vector[1], label=f"Cluster {label} Type {type_val}",
                    color=color, marker=marker, s=100, alpha=0.7)

    plt.title(graph_title, fontsize=16)
    plt.xlabel('Component 1', fontsize=14)
    plt.ylabel('Component 2', fontsize=14)
    plt.grid(True)
    plt.show()


def calculate_cluster_percentages(data):
    cluster_distribution = {}
    for key, value in data.items():
        type_ = key[0]
        cluster = value['cluster']
        if type_ not in cluster_distribution:
            cluster_distribution[type_] = {}
        if cluster not in cluster_distribution[type_]:
            cluster_distribution[type_][cluster] = 0
        cluster_distribution[type_][cluster] += 1

    # Prepare data for DataFrame
    data_for_df = []
    for type_, clusters in cluster_distribution.items():
        for cluster, count in clusters.items():
            data_for_df.append([type_, cluster, count])
            
    # Aggregate totals per type
    type_totals = {}
    for data in data_for_df:
        type_name = data[0]
        count = data[2]
        if type_name in type_totals:
            type_totals[type_name] += count
        else:
            type_totals[type_name] = count

    # Compute totals per cluster per type
    cluster_totals = {}
    for data in data_for_df:
        type_name = data[0]
        cluster_id = data[1]
        count = data[2]
        if type_name not in cluster_totals:
            cluster_totals[type_name] = {}
        if cluster_id not in cluster_totals[type_name]:
            cluster_totals[type_name][cluster_id] = 0
        cluster_totals[type_name][cluster_id] += count

    # Calculate and format the percentage per cluster per type
    output = []
    for type_name, clusters in sorted(cluster_totals.items()):
        output.append(f"Type: {type_name}")
        for cluster_id, count in sorted(clusters.items()):
            percentage = (count / type_totals[type_name]) * 100
            output.append(f"  Cluster {cluster_id}: {percentage:.2f}%")

    return output




# Define color map outside the function
color_map = {
    'gpt-4-turbo-preview': 'yellow',
    'gpt-3.5-turbo': 'red',
    'NYT': 'green',
    'human': 'blue'

}
from sklearn.metrics import precision_score, recall_score, f1_score
def analyze_and_display_clusters(data, num_clusters = 2, title = 'Cluster Distribution'):
    # Aggregate cluster distributions by type
    cluster_distribution = {}
    for key, value in data.items():
        type_ = key[0]
        cluster = value['cluster']
        if type_ not in cluster_distribution:
            cluster_distribution[type_] = {}
        if cluster not in cluster_distribution[type_]:
            cluster_distribution[type_][cluster] = 0
        cluster_distribution[type_][cluster] += 1

    # Prepare data for DataFrame for graph
    data_for_df = []
    for type_, clusters in cluster_distribution.items():
        for cluster, count in clusters.items():
            data_for_df.append([type_, cluster, count])

    # Define AI and human types for the table display
    ai_types = ['gpt-3.5-turbo', 'gpt-4-turbo-preview']
    human_types = ['NYT', 'human']
    cluster_info = {i: {'ai_total': 0, 'human_total': 0, 'total': 0} for i in range(num_clusters)}

    # Process data for AI vs Human content distribution
    for type_, clusters in cluster_distribution.items():
        for cluster, count in clusters.items():
            type_group = 'ai' if type_ in ai_types else 'human'
            cluster_info[cluster][f'{type_group}_total'] += count
            cluster_info[cluster]['total'] += count

    # Determine the majority label in each cluster
    cluster_majority_label = {}
    for cluster_id, info in cluster_info.items():
        if info['ai_total'] >= info['human_total']:
            cluster_majority_label[cluster_id] = 'ai'
        else:
            cluster_majority_label[cluster_id] = 'human'

    # Construct ground truth and predicted labels for evaluation
    y_true = []  # Actual labels
    y_pred = []  # Predicted labels based on majority rule

    for key, value in data.items():
        actual_type = 'ai' if key[0] in ai_types else 'human'
        assigned_cluster = value['cluster']
        predicted_label = cluster_majority_label[assigned_cluster]

        y_true.append(1 if actual_type == 'ai' else 0)  # Convert to binary (AI=1, Human=0)
        y_pred.append(1 if predicted_label == 'ai' else 0)

    # Compute precision, recall, and F1-score
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)


    # DataFrame for table display
    table_data = {
        'Cluster': [],
        'AI Content (%)': [],
        'Human Content (%)': []
    }
    for cluster_id in sorted(cluster_info):
        info = cluster_info[cluster_id]
        ai_percent = (info['ai_total'] / info['total'] * 100) if info['total'] != 0 else 0
        human_percent = (info['human_total'] / info['total'] * 100) if info['total'] != 0 else 0
        table_data['Cluster'].append(cluster_id)
        table_data['AI Content (%)'].append(round(ai_percent, 2))
        table_data['Human Content (%)'].append(round(human_percent, 2))
        
    # Display precision, recall, and F1-score
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # Create and display bar graph
    df_graph = pd.DataFrame(data_for_df, columns=['Type', 'Cluster', 'Count'])
    plt.figure(figsize=(20, 6))
    sns.barplot(x='Cluster', y='Count', hue='Type', data=df_graph, palette=color_map)
    plt.title(title, fontsize=16)
    plt.xlabel('Cluster')
    plt.ylabel('Count')
    plt.legend(title='Type')
    plt.show()

    # Create and display table as an image
    df_table = pd.DataFrame(table_data)
    fig, ax = plt.subplots(figsize=(8, num_clusters * 0.5))  # Adjust size appropriately
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=df_table.values, colLabels=df_table.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)  # Adjust table scale
    plt.show()


In [7]:
# Sample data structure: {(type, id): array_of_floats}

# Fetch the embedded vectors for each article
chatGPT_articles_data = fetch_embedded_vector(chatGPT_articles)
pro_quest_articles_data = fetch_embedded_vector(pro_quest_articles)
nyTimes_articles_data = fetch_embedded_vector(nyTimes_articles)
# getting pos vector
chatGPT_articles_pos = fetch_pos_vector(chatGPT_articles)
pro_quest_articles_pos = fetch_pos_vector(pro_quest_articles)
nyTimes_articles_pos = fetch_pos_vector(nyTimes_articles)
# getting the content and title
chatGPT_articles_content = fetch_title_and_content(chatGPT_articles)
pro_quest_articles_content = fetch_title_and_content(pro_quest_articles)
# Select 200 random articles from pro_quest_articles_content
pro_quest_articles_content = dict(list(pro_quest_articles_content.items())[:200])
nyTimes_articles_content = fetch_title_and_content(nyTimes_articles)

# put all the data together
all_embedded = {**chatGPT_articles_data, **pro_quest_articles_data, **nyTimes_articles_data}
all_pos = {**chatGPT_articles_pos, **pro_quest_articles_pos, **nyTimes_articles_pos}
all_content = {**chatGPT_articles_content, **pro_quest_articles_content, **nyTimes_articles_content}

Get the Data together for the TSNE Algorithm Run

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Get the Data together for the TSNE Algorithm Run

# Extract all texts and fit the vectorizer
all_texts = [value for _, value in all_content.items()]
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_vectorizer.fit(all_texts)  # Fit on all texts to create a consistent vocabulary

# Transform each subset of the data
chatGPT_articles_content_gpt_3_5 = {key: value for key, value in all_content.items() if key[0] == 'gpt-3.5-turbo'}
chatGPT_articles_content_gpt_4 = {key: value for key, value in all_content.items() if key[0] == 'gpt-4-turbo-preview'}
nyTimes_articles_content = {key: value for key, value in all_content.items() if key[0] == 'NYT'}
pro_quest_articles_content = {key: value for key, value in all_content.items() if key[0] == 'human'}
# Use the fitted vectorizer to transform documents
chatGPT_articles_content_gpt_3_5_t = {key: tfidf_vectorizer.transform([value]).toarray() for key, value in chatGPT_articles_content_gpt_3_5.items()}
chatGPT_articles_content_gpt_4_t = {key: tfidf_vectorizer.transform([value]).toarray() for key, value in chatGPT_articles_content_gpt_4.items()}
nyTimes_articles_content_t = {key: tfidf_vectorizer.transform([value]).toarray() for key, value in nyTimes_articles_content.items()}
pro_quest_articles_content_t = {key: tfidf_vectorizer.transform([value]).toarray() for key, value in pro_quest_articles_content.items()}

# Combine all transformed vectors
all_vectors = np.vstack([
    np.vstack(list(chatGPT_articles_content_gpt_3_5_t.values())),
    np.vstack(list(chatGPT_articles_content_gpt_4_t.values())),
    np.vstack(list(nyTimes_articles_content_t.values())),
    np.vstack(list(pro_quest_articles_content_t.values()))
])

# Check the shapes to ensure consistency
print("All vectors shape:", all_vectors.shape)


All vectors shape: (789, 1000)


In [9]:
# Print the first 5 keys of all_embedded
for key in list(all_embedded.keys())[:5]:
    print(key)
    print(len(all_embedded[key]))

('gpt-3.5-turbo', '660c56de05cd8c2a320b49a6')
1536
('gpt-3.5-turbo', '660c56de05cd8c2a320b49aa')
1536
('gpt-3.5-turbo', '660c56de05cd8c2a320b49ad')
1536
('gpt-3.5-turbo', '660c56de05cd8c2a320b49c3')
1536
('gpt-3.5-turbo', '660c56de05cd8c2a320b49c8')
1536


In [10]:
# Perform KMeans clustering in a loop 20 times and run analyze cluster distribution every time
for i in range(20):
    #kmeans_results_emb = perform_kmeans(all_embedded, i+1, 10)
    kmeans_results_pos = perform_kmeans(all_pos, i+1, 10)
    #analyze_and_display_clusters(kmeans_results_emb, i+1, f'KMeans Cluster Distribution using Embedded Vector - Run {i}')
    #print(calculate_cluster_percentages(kmeans_results_emb))
    analyze_and_display_clusters(kmeans_results_pos, i+1, f'KMeans Cluster Distribution using POS Vector - Run {i}')


AttributeError: 'tuple' object has no attribute 'items'

In [ ]:
# Perform KMeans clustering
# 10 and 12 are optimal seed
for seed in range (20):
    kmeans_results_emb = perform_kmeans(all_embedded, 5, seed)
    analyze_and_display_clusters(kmeans_results_emb, 5, 'KMeans Cluster Distribution using Embedded Vector seed = ' + str(seed))
    distribution = calculate_cluster_percentages(kmeans_results_emb)
    for line in distribution:
        print(line)

In [ ]:
kmeans_results_emb = perform_kmeans(all_embedded, 5, 10)
analyze_and_display_clusters(kmeans_results_emb, 5, 'KMeans Cluster Distribution using Embedded Vector')
plot_clusters(kmeans_results_emb, 'KMeans Clustering using Embedded Vector')
distribution = calculate_cluster_percentages(kmeans_results_emb)
for line in distribution:
    print(line)

In [ ]:
# Perform KMeans clustering
kmeans_results_pos = perform_kmeans(all_pos, 8)
plot_clusters(kmeans_results_pos, 'KMeans Clustering using POS Vector')
analyze_and_display_clusters(kmeans_results_pos, 8, 'KMeans Cluster Distribution using POS Vector')

In [ ]:
results = calculate_cluster_percentages(kmeans_results_pos)
for line in results:
    print(line)

In [ ]:
# Perform Hierarchical clustering 40 times and run analyze cluster distribution every time
for i in range(40):
    hierarchical_results_emb = perform_hierarchical(all_embedded, i+1)
    #hierarchical_results_pos = perform_hierarchical(all_pos, i+1)
    analyze_and_display_clusters(hierarchical_results_emb, i+1, f'Hierarchical Cluster Distribution using Embedded Vector - Run {i}')
    #analyze_and_display_clusters(hierarchical_results_pos, i+1, f'Hierarchical Cluster Distribution using POS Vector - Run {i}')

In [ ]:
# Perform Hierarchical clustering
hierarchical_results_emb = perform_hierarchical(all_embedded, 2)
hierarchical_results_pos = perform_hierarchical(all_pos, 2)

In [ ]:
plot_clusters(hierarchical_results_emb, 'Hierarchical Clustering using Embedded Vector')
analyze_and_display_clusters(hierarchical_results_emb, 2, 'Hierarchical Cluster Distribution using Embedded Vector')
plot_clusters(hierarchical_results_pos, 'Hierarchical Clustering using POS Vector')
analyze_and_display_clusters(hierarchical_results_pos, 2, 'Hierarchical Cluster Distribution using POS Vector')

In [ ]:
results = calculate_cluster_percentages(hierarchical_results_emb)
for line in results:
    print(line)

In [ ]:
results = calculate_cluster_percentages(hierarchical_results_pos)
for line in results:
    print(line)

DBSCAN Clustering

In [ ]:
# Perform DBSCAN clustering
dbscan_results_emb = run_DBSCAN(all_embedded, eps=0.5, min_samples=5)
dbscan_results_pos = run_DBSCAN(all_pos, eps=0.5, min_samples=5)

In [ ]:
plot_clusters(dbscan_results_emb, 'DBSCAN Clustering using Embedded Vector')
analyze_and_display_clusters(dbscan_results_emb, 16, 'DBSCAN Cluster Distribution using Embedded Vector')
plot_clusters(dbscan_results_pos, 'DBSCAN Clustering using POS Vector')
analyze_and_display_clusters(dbscan_results_pos, 2, 'DBSCAN Cluster Distribution using POS Vector')

In [ ]:
results = calculate_cluster_percentages(dbscan_results_emb)
for line in results:
    print(line)

In [ ]:
results = calculate_cluster_percentages(dbscan_results_pos)
for line in results:
    print(line)

Gaussian Mixture Models Clustering

In [ ]:
# Perform Gaussian Mixture Models clustering
for i in range(40):
    gmm_results_emb = Gaussian_Mixture_Models(all_embedded, i+1)
    gmm_results_pos = Gaussian_Mixture_Models(all_pos, i+1)
    analyze_and_display_clusters(gmm_results_emb, i+1, f'Gaussian Mixture Models Cluster Distribution using Embedded Vector - Run {i}')
    analyze_and_display_clusters(gmm_results_pos, i+1, f'Gaussian Mixture Models Cluster Distribution using POS Vector - Run {i}')


In [ ]:
gmm_results_emb = Gaussian_Mixture_Models(all_embedded, 8)
gmm_results_pos = Gaussian_Mixture_Models(all_pos, 6)
plot_clusters(gmm_results_emb, 'Gaussian Mixture Models Clustering using Embedded Vector')
analyze_and_display_clusters(gmm_results_emb, 8, 'Gaussian Mixture Models Cluster Distribution using Embedded Vector')
plot_clusters(gmm_results_pos, 'Gaussian Mixture Models Clustering using POS Vector')
analyze_and_display_clusters(gmm_results_pos, 6, 'Gaussian Mixture Models Cluster Distribution using POS Vector')

In [ ]:
results = calculate_cluster_percentages(gmm_results_emb)
for line in results:
    print(line)

In [ ]:
results = calculate_cluster_percentages(gmm_results_pos)
for line in results:
    print(line)

Isolation Forest Clustering

In [ ]:
for i in range(40):
    isolation_forest_results_emb = Isolation_Forest(all_embedded, i+1)
    isolation_forest_results_pos = Isolation_Forest(all_pos, i+1)
    results = calculate_cluster_percentages(isolation_forest_results_emb)
    print (f'Isolation Forest Cluster Distribution using Embedded Vector - Run {i}')
    for line in results:
        print(line)
    results = calculate_cluster_percentages(isolation_forest_results_pos)
    print (f'Isolation Forest Cluster Distribution using POS Vector - Run {i}')
    for line in results:
        print(line)


In [ ]:
# Perform Isolation Forest clustering
isolation_forest_results_emb = Isolation_Forest(all_embedded, 10)
isolation_forest_results_pos = Isolation_Forest(all_pos, 10)

In [ ]:
plot_clusters(isolation_forest_results_emb, 'Isolation Forest Clustering using Embedded Vector')
analyze_and_display_clusters(isolation_forest_results_emb, 10, 'Isolation Forest Cluster Distribution using Embedded Vector')
plot_clusters(isolation_forest_results_pos, 'Isolation Forest Clustering using POS Vector')
analyze_and_display_clusters(isolation_forest_results_pos, 10, 'Isolation Forest Cluster Distribution using POS Vector')

In [ ]:
results = calculate_cluster_percentages(isolation_forest_results_emb)
for line in results:
    print(line)

In [ ]:
results = calculate_cluster_percentages(isolation_forest_results_pos)
for line in results:
    print(line)

TSNE Clustering

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Assuming all_vectors and labels are already defined
labels_array = (['GPT-3.5'] * len(chatGPT_articles_content_gpt_3_5_t) +
                ['GPT-4'] * len(chatGPT_articles_content_gpt_4_t) +
                ['NYTimes'] * len(nyTimes_articles_content_t) +
                ['ProQuest'] * len(pro_quest_articles_content_t))

# Colors for different labels
colors = {'GPT-3.5': 'blue', 'GPT-4': 'red', 'NYTimes': 'green', 'ProQuest': 'yellow'}
color_list = [colors[label] for label in labels_array]

# Define perplexities and learning rates to test
perplexities = [5, 30, 50]  # Small, medium, large
learning_rates = [10, 200, 1000]  # Low, medium, high

# Prepare the figure for subplots
fig, axs = plt.subplots(len(perplexities), len(learning_rates), figsize=(15, 12))

for i, perp in enumerate(perplexities):
    for j, lr in enumerate(learning_rates):
        tsne = TSNE(n_components=2, perplexity=perp, learning_rate=lr, n_iter=3000, random_state=42)
        X_tsne = tsne.fit_transform(all_vectors)

        axs[i, j].scatter(X_tsne[:, 0], X_tsne[:, 1], c=color_list, alpha=0.5)
        axs[i, j].set_title(f'Perplexity: {perp}, LR: {lr}')
        axs[i, j].set_xticks([])
        axs[i, j].set_yticks([])

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

